# SP03 — Sub-cellular Resolution: seqFISH+ / Xenium / MERFISH

**Tools:** `squidpy`, `spatialdata`  
**Dataset:** Mouse brain seqFISH+ (~19k cells, `sq.datasets.seqfish()`)  
**Papers:** [Eng et al. 2019, Nature (seqFISH+)](https://doi.org/10.1038/s41586-019-1049-y) · [Chen et al. 2015, Science (MERFISH)](https://doi.org/10.1126/science.aaa6090)

---

## Visium vs. sub-cellular resolution platforms

Visium spots contain 5–30 cells. You never see a single cell — only its contribution to a mixed spot. Sub-cellular resolution platforms measure **individual transcripts at known positions** inside each cell:

```
Visium:                    seqFISH+ / Xenium / MERFISH:

[spot]                     [cell polygon]
 ●────────────              ╔══════════════╗
 │ cell A                  ║  ●  ●    ●   ║  ← individual transcripts
 │ cell B     →            ║     ●  ●     ║
 │ cell C                  ║  ●       ●   ║
 └────────────              ╚══════════════╝
 ~55 µm spot                ~10-20 µm cell
```

**What changes:**
- Data model: **transcript coordinates** (x, y, gene) instead of spot × gene matrix
- Cell segmentation required: transcripts must be assigned to cells
- Much higher spatial resolution: can resolve cell-cell interactions at micron scale
- Gene panel: typically 100–10,000 genes (Xenium: 280–5000; MERFISH: 500–10,000; seqFISH+: ~10,000)
- Much larger files: millions of transcript detections

**Platforms compared:**

| Platform | Resolution | Genes | Cells/FOV | Key feature |
|----------|-----------|-------|----------|-------------|
| seqFISH+ | ~100 nm | ~10,000 | ~1,000 | Sequential hybridization |
| MERFISH (Vizgen) | ~100 nm | ~1,000 | ~50,000 | Error-robust barcoding |
| Xenium (10x) | ~100 nm | 280–5,000 | whole tissue | Integrated cell segmentation |
| CosMx (NanoString) | ~100 nm | ~6,000 | whole tissue | Subcellular + protein |

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection

sc.settings.set_figure_params(dpi=80, facecolor='white')
print(f'squidpy {sq.__version__}')

## 1. Load seqFISH+ Data

The squidpy seqFISH+ dataset is a mouse brain section with:
- ~19,416 segmented cells
- 351 genes measured
- Single-cell resolution with spatial coordinates
- Pre-segmented (transcripts already assigned to cells)

In [ ]:
# Auto-downloads ~50 MB on first run
adata = sq.datasets.seqfish()
print(adata)
print('\nobs columns:', adata.obs.columns.tolist())
print('obsm keys:', list(adata.obsm.keys()))
print('\nCell type distribution:')
print(adata.obs['celltype_mapped_refined'].value_counts().head(10))

In [ ]:
# The key difference from Visium: coordinates are cell centroids (not spots on a grid)
# Resolution is single-cell: each row = one segmented cell
print('Spatial coordinate range:')
coords = adata.obsm['spatial']
print(f'  X: {coords[:,0].min():.0f} — {coords[:,0].max():.0f} µm')
print(f'  Y: {coords[:,1].min():.0f} — {coords[:,1].max():.0f} µm')
print(f'\nCell density: {adata.n_obs / ((coords[:,0].ptp() * coords[:,1].ptp()) / 1e6):.0f} cells/mm²')

## 2. Data Visualization at Single-Cell Resolution

At this resolution, individual cells are visible — visualizations look very different from Visium spot plots.

In [ ]:
# Cell type map at single-cell resolution
sq.pl.spatial_scatter(
    adata,
    color='celltype_mapped_refined',
    size=3,     # smaller dots — these are actual cells, not ~55µm spots
    alpha=0.7,
    figsize=(8, 6),
    title='Mouse brain seqFISH+ — cell type map',
)

In [ ]:
# Gene expression at single-cell resolution
# Classic cell-type markers should show tight spatial patterns
marker_genes = ['Camk2a', 'Gad1', 'Mbp', 'Aqp4', 'Cx3cr1']
available = [g for g in marker_genes if g in adata.var_names]

if available:
    sq.pl.spatial_scatter(
        adata, color=available,
        size=2.5, ncols=3, figsize=(12, 8),
        title=available,
    )
else:
    print('Marker genes not in panel; printing available genes:')
    print(adata.var_names[:20].tolist())

In [ ]:
# Zoom in to a small region to see individual cell resolution
# Select a 500×500 µm region in the center of the tissue
center_x = np.median(adata.obsm['spatial'][:, 0])
center_y = np.median(adata.obsm['spatial'][:, 1])
window = 250  # µm

cells_in_window = (
    (adata.obsm['spatial'][:, 0] > center_x - window) &
    (adata.obsm['spatial'][:, 0] < center_x + window) &
    (adata.obsm['spatial'][:, 1] > center_y - window) &
    (adata.obsm['spatial'][:, 1] < center_y + window)
)
adata_zoom = adata[cells_in_window].copy()
print(f'Cells in 500×500 µm window: {adata_zoom.n_obs}')

sq.pl.spatial_scatter(
    adata_zoom,
    color='celltype_mapped_refined',
    size=20,    # larger dots since we're zoomed in
    figsize=(6, 6),
    title=f'500×500 µm window ({adata_zoom.n_obs} cells)',
)

## 3. Single-Cell Spatial Analysis

With true single-cell resolution, all the analyses from theis_ecosystem/T02 become more powerful because you're working with actual cells, not mixed spots.

In [ ]:
# Build spatial kNN graph at single-cell scale
# n_neighs=10 for single cells (vs. 6 for Visium hexagonal grid)
# delaunay=True: compute a Delaunay triangulation for better neighbor definition
sq.gr.spatial_neighbors(
    adata,
    coord_type='generic',   # not a hexagonal grid — use generic Euclidean
    n_neighs=10,
    delaunay=True,
)
print('Spatial graph:', adata.obsp['spatial_connectivities'].shape)
print('Mean neighbors per cell:',
      (adata.obsp['spatial_connectivities'] > 0).sum(axis=1).mean())

In [ ]:
# Neighborhood enrichment: which cell types are spatially adjacent?
sq.gr.nhood_enrichment(adata, cluster_key='celltype_mapped_refined', n_perms=100, seed=42)
sq.pl.nhood_enrichment(
    adata,
    cluster_key='celltype_mapped_refined',
    title='Cell type co-occurrence (seqFISH+)',
    figsize=(8, 6),
)

In [ ]:
# Spatially variable genes (SVGs) at single-cell resolution
# With ~20k cells this is much more powered than ~2k Visium spots
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

sq.gr.spatial_autocorr(
    adata,
    mode='moran',
    genes=adata.var_names,  # all 351 genes in panel
    n_perms=100,
    n_jobs=1,
)
moranI = adata.uns['moranI'].sort_values('I', ascending=False)
print(f'Significant SVGs (padj<0.05): {(moranI["pval_norm_fdr_bh"] < 0.05).sum()} / {len(moranI)}')
print('\nTop 10 SVGs:')
print(moranI.head(10)[['I', 'pval_norm_fdr_bh']])

## 4. Working with Xenium Data (SpatialData format)

Xenium output includes **individual transcript coordinates** — a fundamentally different data model. Each detection is a row with (x, y, z, gene) rather than a cell-by-gene matrix.

In [ ]:
# Xenium data structure (conceptual walkthrough)
# In practice, load with: sdata = spatialdata_io.xenium('path/to/xenium_output/')
#
# A Xenium SpatialData object contains:
#   sdata.points['transcripts'] — per-transcript DataFrame:
#     x_location, y_location, z_location, feature_name, qv (quality value)
#     Typically 10M–100M rows for a full tissue section
#
#   sdata.shapes['cell_boundaries'] — GeoDataFrame with polygon per cell
#   sdata.shapes['nucleus_boundaries'] — nuclear polygons
#   sdata.labels['cell_labels'] — raster segmentation mask
#   sdata.tables['table'] — cell-by-gene AnnData (pre-aggregated by 10x)

print('Xenium data model:')
print('  sdata.points["transcripts"]:   x, y, z, gene, quality per transcript')
print('  sdata.shapes["cell_boundaries"]: polygon per cell')
print('  sdata.shapes["nucleus_boundaries"]: nuclear polygon per cell')
print('  sdata.labels["cell_labels"]:   raster segmentation')
print('  sdata.tables["table"]:          cell × gene AnnData (aggregated)')
print()
print('Key difference from Visium:')
print('  Visium: spot_id → counts (pre-aggregated, no sub-spot resolution)')
print('  Xenium: each transcript at (x, y, z) → assign to cell → aggregate')

In [ ]:
# Simulate a Xenium-style transcript DataFrame to show the data model
np.random.seed(42)

# Generate 50k mock transcript detections in a 1mm × 1mm field
n_transcripts = 50_000
genes = ['Camk2a', 'Gad1', 'Mbp', 'Aqp4', 'Cx3cr1'] * (n_transcripts // 5)
transcripts_df = pd.DataFrame({
    'x_location': np.random.uniform(0, 1000, n_transcripts),
    'y_location': np.random.uniform(0, 1000, n_transcripts),
    'z_location': np.random.uniform(0, 10, n_transcripts),
    'feature_name': genes,
    'qv': np.random.uniform(20, 40, n_transcripts),  # quality score
})

print('Example Xenium transcript table:')
print(transcripts_df.head(10))
print(f'\nTotal: {len(transcripts_df):,} transcripts')
print(f'Unique genes: {transcripts_df["feature_name"].nunique()}')

In [ ]:
# Transcript density plot: where are transcripts physically located?
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# All transcripts — density
axes[0].hist2d(
    transcripts_df['x_location'],
    transcripts_df['y_location'],
    bins=50, cmap='magma',
)
axes[0].set_title('Transcript density (all genes)')
axes[0].set_xlabel('x (µm)')
axes[0].set_ylabel('y (µm)')

# Per-gene spatial patterns
for gene in ['Camk2a', 'Gad1', 'Mbp']:
    gene_df = transcripts_df[transcripts_df['feature_name'] == gene]
    axes[1].scatter(gene_df['x_location'], gene_df['y_location'],
                    s=0.5, alpha=0.3, label=gene)
axes[1].set_title('Per-gene transcript locations')
axes[1].legend(markerscale=10)
plt.tight_layout()
plt.show()

In [ ]:
# Cell segmentation: assign transcripts to cells
# In real Xenium, 10x provides pre-segmented boundaries
# The typical workflow after loading:
#
# 1. Load: sdata = spatialdata_io.xenium('output/')
# 2. Cell-level table already aggregated: sdata.tables['table']
# 3. For transcript-level analysis:
#    transcripts = sdata.points['transcripts'].compute()  # Dask → pandas
#    cell_shapes = sdata.shapes['cell_boundaries']        # GeoDataFrame
#    # Spatial join: assign transcript to cell
#    import geopandas as gpd
#    trans_gdf = gpd.GeoDataFrame(transcripts,
#        geometry=gpd.points_from_xy(transcripts.x_location, transcripts.y_location))
#    assigned = trans_gdf.sjoin(cell_shapes, how='left', predicate='within')

print('Xenium analysis workflow summary:')
print('1. sdata = spatialdata_io.xenium("output/")')
print('2. adata = sdata.tables["table"]  # cell × gene AnnData')
print('3. Run standard scanpy + squidpy pipeline on adata')
print('4. For sub-cellular: sdata.points["transcripts"] + polygon spatial join')

## 5. Comparing seqFISH+ to Visium: What Changes?

Same biological question, different resolution. Let's compare spatial autocorrelation results.

In [ ]:
# Compare Moran's I from seqFISH+ (single cell) vs. Visium (spots)
if os.path.exists('../data/visium_processed.h5ad'):
    import os
    adata_vis = sc.read_h5ad('../data/visium_processed.h5ad')
    
    # Get shared genes
    shared = list(set(adata.var_names) & set(adata_vis.var_names))
    print(f'Genes in seqFISH+: {adata.n_vars}  |  In Visium: {adata_vis.n_vars}  |  Shared: {len(shared)}')
    
    if len(shared) > 10:
        moranI_seqfish = adata.uns['moranI'].loc[
            adata.uns['moranI'].index.isin(shared), 'I'
        ].rename('seqFISH+')
        
        moranI_visium = adata_vis.uns.get('moranI', pd.DataFrame())
        if not moranI_visium.empty:
            moranI_comp = pd.concat([moranI_seqfish,
                                     moranI_visium.loc[moranI_visium.index.isin(shared), 'I'].rename('Visium')],
                                    axis=1).dropna()
            print(f'\nMoran\'s I comparison ({len(moranI_comp)} shared genes):')
            print(moranI_comp.describe())
else:
    print('Run SP01 first to generate ../data/visium_processed.h5ad')
    print('Key comparison: seqFISH+ Moran\'s I is generally higher (more cells = more power)')

---
## Summary

**Key differences from Visium:**

| | Visium | seqFISH+ / Xenium / MERFISH |
|-|--------|-----------------------------|
| Resolution | ~55 µm spots | ~100 nm (single transcript) |
| Cell segmentation | Not needed (spots) | Required or pre-done |
| Gene panel | Whole transcriptome | 100–10,000 targeted genes |
| Data model | Spot × gene matrix | Transcript (x, y, z, gene) → aggregate to cell × gene |
| `squidpy` graph | `coord_type='visium'`, n_neighs=6 | `coord_type='generic'`, delaunay=True |
| SpatialData | `sdata.tables['table']` | `sdata.points['transcripts']` + `sdata.shapes['cell_boundaries']` |

**Load Xenium:**
```python
from spatialdata_io import xenium
sdata = xenium('path/to/xenium_output/', dataset_id='sample1')
adata = sdata.tables['table']  # standard cell × gene AnnData
transcripts = sdata.points['transcripts'].compute()  # lazy → pandas
```

**Next:** SP04 — Multi-sample spatial integration (aligning multiple slides, batch effects across donors).